# ⚠️ Setup Instructions

To run this notebook successfully, please ensure you complete the following steps:

1. **Switch to T4 GPU**: Ensure your runtime is configured to use a T4 GPU. In the menu, go to **Runtime > Change runtime type**, select **T4 GPU** under *Hardware accelerator*, and click **Save**.
2. **Shared Folder Shortcut**: Make sure you have access to the shared repository Drive folder and have created a **shortcut** from it directly to your own Google Drive's home directory (`My Drive`). This allows the code to resolve the path `/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis` correctly.
3. **Drive Permissions**: When you select **Runtime > Run all**, Google Colab will prompt you for Google Drive access permissions. You must authorize full access to your Drive for the file reading and writing processes to run without interruption.

# Traffic Risk Agent Pipeline
This notebook walks you through setting up and running the Traffic Risk Agent Pipeline using YOLO spatial metadata extraction and an Ollama-hosted vision-language model (Moondream) to assess traffic scene risk.

## Cell 1: Team Instructions & Drive Mount
We will mount Google Drive, check GPU status, and configure the working directory to access our shared repository.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Check GPU availability
!nvidia-smi

# Set working directory to project repository
REPO_PATH = '/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis'
os.chdir(REPO_PATH)
print(f"Current Working Directory: {os.getcwd()}")

# List directory contents
print("Directory contents:")
print(os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fri Oct 16 09:52:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [ ]:
# ── Utility: Generate / Reset Ground Truth Table ───────────────────────
# Uncomment the line below ONLY when you want to scan data/raw_images and
# pre-populate / reset data/ground_truth.csv with fresh assignments:

# !python src/utils/build_ground_truth.py


## Cell 2: Install & Start Ollama Background Daemon
We will install Ollama, launch it as a background service, wait for initialization, and pull the `moondream` vision model.

In [ ]:
import os
import subprocess
import time
import urllib.request

# 1. Install/Update hardware utilities and latest Ollama binary
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Kill stale Ollama daemon (exact process name only, preventing subshell kills)
!pkill -9 -x ollama || true
time.sleep(2)

# 3. Check installed Ollama version (must be >= 0.4.0)
!/usr/local/bin/ollama --version

# 4. Set CUDA & concurrency environment variables
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_FLASH_ATTENTION"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "10m"

# 5. Start a FRESH Ollama daemon in the background with GPU support
print("Starting fresh Ollama daemon with GPU support...")
process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 6. Wait for the server to bind to port 11434
print("Waiting for Ollama server to become available...")
for i in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama server is active and running successfully with GPU support!")
        break
    except Exception:
        time.sleep(2)
else:
    print("⚠️ Ollama daemon failed to start. Check daemon logs.")

# 7. Pull primary model (gemma3:12b)
print("Pulling model...")
# !/usr/local/bin/ollama pull gemma3:12b

# To pull Qwen VL for the follow-up comparison, uncomment:
# !/usr/local/bin/ollama pull qwen2.5vl:7b
!/usr/local/bin/ollama pull qwen3-vl:8b-instruct

!/usr/local/bin/ollama list


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 3,917 B in 2s (2,323 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependen

## Cell 3: Install Python Dependencies
We will install the required Python libraries using standard Colab pip commands.

In [ ]:
# Fallback for uv package manager:
# !pip install uv && !uv sync

!pip install ultralytics opencv-python pandas scipy scikit-learn ollama

## Cell 4: Single-Image Sanity Check Test
We will run a dry-run test on a sample image to verify YOLO extraction and Ollama VLM responses.

In [ ]:
# import cv2
# import glob
# from ultralytics import YOLO
# import ollama

# # Find a sample image
# image_files = glob.glob('data/raw_images/*')
# if not image_files:
#     print("No images found in data/raw_images/. Please check paths.")
# else:
#     sample_img_path = image_files[0]
#     print(f"Testing with image: {sample_img_path}")

#     # 1. Run YOLO Object Detection
#     yolo_model = YOLO('yolo26n.pt')
#     results = yolo_model(sample_img_path)
#     boxes = results[0].boxes
#     detected_objects = [yolo_model.names[int(cls)] for cls in boxes.cls]
#     print(f"Detected objects: {detected_objects}")

#     # 2. Query Ollama llama3.2-vision Model
#     prompt = "Analyze the traffic risk of this scene. Provide a JSON output with keys: 'Confidence' (number 0-100) and 'Meaning' (risk description)."
#     try:
#         response = ollama.generate(
#             model='llava:7b',
#             prompt=prompt,
#             images=[sample_img_path]
#         )
#         print("\nOllama Response:")
#         print(response['response'])
#     except Exception as e:
#         print(f"Error connecting to Ollama API: {e}")

## Cell 5: Full Pipeline Execution
Now we will run the main evaluation pipeline. Results are written directly to `output/experiment_results.csv` on your persistent Drive storage.

In [ ]:
GT_URL = "https://docs.google.com/spreadsheets/d/13Ldjb5q13H5s9ip-RqOCp8QufUqQE-g6ltsinGnGnkc/edit?usp=sharing"

# !python main.py --provider gemini --gt-url "$GT_URL"


# Run the 10-image smoke test:
# !python main.py --provider ollama --model gemma3:12b --limit 10
!python main.py --provider ollama --model qwen3-vl:8b-instruct --gt-url "$GT_URL" --limit 10
#!python main.py --provider ollama --model qwen2.5vl:7b

# Once verified, run the full 58 images by removing --limit 10:
# !python main.py --provider ollama --model gemma3:12b


Limiting execution to first 10 of 58 image(s).

Processing 10 image(s) from /content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis/data/raw_images

⬇️  Downloading ground truth from Google Sheets …
   ✓ Downloaded 58 rows.

Loaded 15 ground-truth entries from https://docs.google.com/spreadsheets/d/13Ldjb5q13H5s9ip-RqOCp8QufUqQE-g6ltsinGnGnkc/edit?usp=sharing.

Initialising YOLO perception extractor …
Initialising VLM client (ollama) …

[1/10] Processing: 0.png
  YOLO: 4 road user(s) detected
  VLM Condition A (image-only) …
    → Attention Score: 49/100 [Stay Alert] (29.58s)
      • traffic_density       :  2/10 | Only a few pedestrians and one cyclist are visible, indicating low vehicle density.
      • vulnerable_road_users :  5/10 | Multiple pedestrians are walking along the street, and a cyclist is nearby, requiring cautious navigation.
      • conflict_proximity    :  6/10 | Pedestrians are crossing or standing near the tram tracks, and a cyclist is cl

In [ ]:
# !nvidia-smi

## Cell 6: Results Visualization & Evaluation Metrics
Finally, we will load the pipeline run results, compare predictions with ground truth, and compute evaluation metrics (Pearson, Spearman, and Classification scores).

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import precision_score, recall_score, f1_score

results_path = 'output/experiment_results.csv'
try:
    df = pd.read_csv(results_path)
    print("Preview of experiment results:")
    display(df.head())

    if 'ground_truth' in df.columns and 'predicted_risk' in df.columns:
        # Continuous Evaluation Metrics
        r_coef, _ = pearsonr(df['predicted_risk'], df['ground_truth'])
        rho_coef, _ = spearmanr(df['predicted_risk'], df['ground_truth'])
        print(f"\nPearson Correlation (r): {r_coef:.3f}")
        print(f"Spearman Rank Correlation (rho): {rho_coef:.3f}")

        # Classification Evaluation Metrics (Threshold >= 70)
        threshold = 70
        y_true = (df['ground_truth'] >= threshold).astype(int)
        y_pred = (df['predicted_risk'] >= threshold).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        print(f"\nClassification metrics (Threshold >= {threshold}):")
        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1 Score:  {f1:.3f}")
    else:
        print("Columns 'ground_truth' or 'predicted_risk' missing from the results csv.")
except FileNotFoundError:
    print(f"Results file not found at {results_path}. Ensure the pipeline completed successfully.")

Preview of experiment results:


,image_id,yolo_detections,yolo_summary,condition_a_score,condition_a_meaning,condition_a_state,condition_a_time_sec,condition_a_raw,condition_b_score,condition_b_meaning,...,cond_b_conflict_proximity_score,cond_b_conflict_proximity_reason,cond_a_road_complexity_score,cond_a_road_complexity_reason,cond_b_road_complexity_score,cond_b_road_complexity_reason,cond_a_impaired_visibility_score,cond_a_impaired_visibility_reason,cond_b_impaired_visibility_score,cond_b_impaired_visibility_reason
0,0,4,Detected Road Users: 4 pedestrians.\n - 1 ped...,49,Traffic density (2/10): Only a few pedestrians...,Stay Alert,29.58,"{\n ""traffic_density"": {""reason"": ""Only a few...",36,Vulnerable road users (5/10): Multiple pedestr...,...,3,"Pedestrians are walking near the tram tracks, ...",4,The street features tram tracks and pedestrian...,2,The street features tram tracks and standard u...,1,"The scene is well-lit with clear daylight, and...",0,The scene is well-lit with clear daylight and ...
1,1,6,"Detected Road Users: 4 cars, 2 pedestrians.\n ...",20,Traffic density (3/10): Several vehicles are v...,Relax,7.42,"{\n ""traffic_density"": {""reason"": ""Several ve...",22,Traffic density (3/10): Multiple cars are visi...,...,1,No vehicles or obstacles are in immediate conf...,2,The road is straight with clear lane markings ...,1,"The road is a straight, multi-lane highway wit...",2,"The scene is bright with clear visibility, tho...",2,"The scene is well-lit with clear visibility, t..."
2,10,7,Detected Road Users: 7 cars.\n - 1 car (large...,17,Traffic density (3/10): Multiple cars are park...,Relax,8.23,"{\n ""traffic_density"": {""reason"": ""Multiple c...",20,Traffic density (3/10): Multiple cars are park...,...,2,The closest vehicle is parked on the right sid...,1,"The road is a straight, two-lane street with n...",1,"The road is straight with no intersections, ro...",0,The scene is clearly visible under bright dayl...,0,The scene is captured under clear daylight wit...
3,100,0,Detected Road Users: None.,27,Traffic density (2/10): Only one distant red t...,Stay Alert,7.88,"{\n ""traffic_density"": {""reason"": ""Only one d...",26,Traffic density (2/10): Only one distant truck...,...,2,The truck is far ahead and does not pose an im...,4,"The road is straight with clear lane markings,...",4,The road is straight with clear lane markings ...,0,"The scene is well-lit with clear skies, offeri...",0,"The scene is well-lit with clear skies, offeri..."
4,102,4,Detected Road Users: 4 cars.\n - 1 car (small...,29,Traffic density (4/10): Several vehicles are p...,Stay Alert,7.78,"{\n ""traffic_density"": {""reason"": ""Several ve...",22,Traffic density (3/10): Multiple vehicles are ...,...,2,No vehicles are in immediate conflict or cross...,3,The scene includes a standard intersection wit...,3,The scene shows a standard intersection with c...,0,"The scene is well-lit with clear visibility, n...",0,The scene is well-lit with clear visibility an...


Columns 'ground_truth' or 'predicted_risk' missing from the results csv.
